# RAG Assignment: Baseline (Bi-Encoder) vs. Re-Ranking (Cross-Encoder)

**Document:** *D&D Basic Rules (2018, v1.0)*, a 180-page rulebook with lots of general rules, exceptions, conditions and look-alike passages (monster stat blocks, spell descriptions, flavour text).

**Pipelines (same chunk corpus for both):**

| | Pipeline 1: Baseline | Pipeline 2: Re-Ranking |
|---|---|---|
| Stage 1 | Bi-encoder `all-MiniLM-L6-v2` → FAISS inner product (cosine) → Top 10 | *same Top 10* |
| Stage 2 | take Top 3 as-is | Cross-encoder `ms-marco-MiniLM-L-6-v2` scores all 10 pairs → re-sort → Top 3 |
| Generation | `gemini-flash-lite-latest`, identical prompt | `gemini-flash-lite-latest`, identical prompt |

Notebook layout:
1. Setup
2. PDF loading & cleaning
3. Chunking
4. Bi-encoder index (Pipeline 1 retrieval)
5. Cross-encoder re-ranker (Pipeline 2)
6. LLM + shared prompt
7. Failure-mode stress tests (rank shifts + Answer 1 vs Answer 2)
8. Small retrieval evaluation (Hit@3 / MRR)
9. Latency profiling
10. Scaling experiment (why not cross-encode the whole corpus?)

## 1. Setup

In [1]:
# Standard libs
import re, time, statistics, warnings
warnings.filterwarnings("ignore")

# Data / numerics
import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", 120)

# PDF parsing + chunking
import pymupdf
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Retrieval models + vector store
import torch
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

# LLM (Gemini via LangChain). GOOGLE_API_KEY is read from the .env file.
from dotenv import load_dotenv, find_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv(find_dotenv(usecwd=True))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


## 2. Load and clean the PDF

Every page has a running header (page number, chapter name) and a legal footer line. If left in, *every* chunk shares that text, which adds noise to the embeddings, so it is stripped.
Curly quotes are normalised to plain quotes and hard line-wraps are joined back into sentences.
Pages 1–2 (credits, table of contents) and the character sheet / back cover at the end are skipped.

In [2]:
PDF_PATH = "DnD_BasicRules_2018.pdf"
BOILERPLATE = ("D&D Basic Rules (Version 1.0). Not for resale. "
               "Permission granted to print and photocopy this document for personal use only.")

def clean_page(text: str) -> str:
    # remove footer + normalise quotes
    text = text.replace(BOILERPLATE, "")
    text = (text.replace("\u2019", "'").replace("\u2018", "'")
                .replace("\u201c", '"').replace("\u201d", '"'))
    # drop the running header (page number lines / "Chapter X: ..." in the first few lines)
    lines = text.split("\n")
    kept = []
    for i, line in enumerate(lines):
        s = line.strip()
        if i < 4 and (re.fullmatch(r"\d+", s) or s.startswith(("Chapter ", "Appendix "))):
            continue
        kept.append(line)
    text = "\n".join(kept)
    text = re.sub(r"-\s*\n(?=[a-z])", "", text)   # re-join hyphenated words split across lines
    text = re.sub(r"[ \t]*\n[ \t]*", " ", text)    # join wrapped lines
    return re.sub(r"\s{2,}", " ", text).strip()

doc = pymupdf.open(PDF_PATH)
pages = [(i + 1, clean_page(p.get_text())) for i, p in enumerate(doc) if 2 <= i <= 175]
print(f"PDF pages: {doc.page_count} | pages kept: {len(pages)} | characters: {sum(len(t) for _, t in pages):,}")
print("\nSample (page 79):\n", dict(pages)[79][:500])

PDF pages: 180 | pages kept: 174 | characters: 727,012

Sample (page 79):
 Falling Unconscious If damage reduces you to 0 hit points and fails to kill you, you fall unconscious (see appendix A). This unconsciousness ends if you regain any hit points. Death Saving Throws Whenever you start your turn with 0 hit points, you must make a special saving throw, called a death saving throw, to determine whether you creep closer to death or hang onto life. Unlike other saving throws, this one isn't tied to any ability score. You are in the hands of fate now, aided only by spell


## 3. Chunking

`RecursiveCharacterTextSplitter` with ~800 characters and 100 characters overlap, splitting on sentence boundaries first.
Chunks are created per page so each chunk keeps its page number (useful for citing / checking answers).
Tiny fragments (< 80 chars) are dropped. **Both pipelines use exactly this chunk list.**

In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800, chunk_overlap=100,
    separators=[". ", " ", ""], keep_separator="end",
)

chunks = []
for page_no, text in pages:
    for piece in splitter.split_text(text):
        if len(piece) > 80:
            chunks.append({"id": len(chunks), "page": page_no, "text": piece})

lengths = [len(c["text"]) for c in chunks]
print(f"Total chunks: {len(chunks)} | avg length: {np.mean(lengths):.0f} chars | max: {max(lengths)}")
chunks[552]

Total chunks: 1113 | avg length: 684 chars | max: 799


{'id': 552,
 'page': 79,
 'text': "Falling Unconscious If damage reduces you to 0 hit points and fails to kill you, you fall unconscious (see appendix A). This unconsciousness ends if you regain any hit points. Death Saving Throws Whenever you start your turn with 0 hit points, you must make a special saving throw, called a death saving throw, to determine whether you creep closer to death or hang onto life. Unlike other saving throws, this one isn't tied to any ability score. You are in the hands of fate now, aided only by spells and features that improve your chances of succeeding on a saving throw. Roll a d20. If the roll is 10 or higher, you succeed. Otherwise, you fail. A success or failure has no effect by itself. On your third success, you become stable (see below). On your third failure, you die."}

## 4. Pipeline 1 retrieval: Bi-Encoder + FAISS

The bi-encoder embeds every chunk **once, offline**. Vectors are L2-normalised, so the inner product in `IndexFlatIP` is the cosine similarity.
At query time only the query is embedded, then one vector search returns the Top 10.

In [4]:
BI_ENCODER_NAME = "sentence-transformers/all-MiniLM-L6-v2"
bi_encoder = SentenceTransformer(BI_ENCODER_NAME, device=DEVICE)

t0 = time.perf_counter()
chunk_embeddings = bi_encoder.encode(
    [c["text"] for c in chunks], batch_size=64,
    normalize_embeddings=True, show_progress_bar=False,
).astype("float32")
index_time = time.perf_counter() - t0

index = faiss.IndexFlatIP(chunk_embeddings.shape[1])   # exact inner-product search
index.add(chunk_embeddings)
print(f"Indexed {index.ntotal} chunks, dim={chunk_embeddings.shape[1]}, offline indexing took {index_time:.2f}s")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Indexed 1113 chunks, dim=384, offline indexing took 3.06s


In [5]:
TOP_K_RETRIEVE = 10
TOP_K_CONTEXT = 3

def bi_encoder_retrieve(query: str, k: int = TOP_K_RETRIEVE):
    # embed the query and return the k nearest chunks as [(chunk_id, cosine_score), ...]
    q_vec = bi_encoder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q_vec, k)
    return [(int(i), float(s)) for i, s in zip(ids[0], scores[0])]

bi_encoder_retrieve("How does grappling work?")[:3]

[(542, 0.5471441745758057),
 (543, 0.4896976053714752),
 (794, 0.4318876564502716)]

## 5. Pipeline 2: Cross-Encoder re-ranker

The cross-encoder reads the query and the chunk **together** in one transformer pass and outputs one relevance score per pair.
It only re-scores the 10 candidates from stage 1. It never sees the rest of the corpus.

In [6]:
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME, device=DEVICE)

def cross_encoder_rerank(query: str, candidates):
    # candidates: [(chunk_id, bi_score), ...] in baseline order
    # returns list of dicts sorted by cross-encoder score (descending)
    pairs = [(query, chunks[cid]["text"]) for cid, _ in candidates]
    ce_scores = cross_encoder.predict(pairs)
    rows = [
        {"chunk_id": cid, "baseline_rank": r + 1, "bi_score": bi, "ce_score": float(ce)}
        for r, ((cid, bi), ce) in enumerate(zip(candidates, ce_scores))
    ]
    rows.sort(key=lambda x: x["ce_score"], reverse=True)
    for new_rank, row in enumerate(rows, start=1):
        row["rerank_rank"] = new_rank
    return rows

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

## 6. LLM and the shared prompt

Both pipelines use the **identical** prompt template and the same model at temperature 0, so the only difference between Answer 1 and Answer 2 is the 3 chunks in the context.
The prompt tells the model to answer only from the context, so missing context should show up as "not in context", a wrong rule, or a hallucination.

In [7]:
llm = ChatGoogleGenerativeAI(model="gemini-flash-lite-latest", temperature=0)

PROMPT_TEMPLATE = (
    "You are a rules assistant for the Dungeons & Dragons Basic Rules.\n"
    "Answer the question using ONLY the context passages below. "
    "If the context does not contain the answer, say so. "
    "Mention any exception or restriction that applies. Keep the answer to 2-4 sentences.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n"
    "Answer:"
)

def build_context(chunk_ids):
    return "\n\n".join(f"[Passage {n} | page {chunks[c]['page']}]\n{chunks[c]['text']}"
                       for n, c in enumerate(chunk_ids, start=1))

def generate_answer(question, chunk_ids, retries=3):
    prompt = PROMPT_TEMPLATE.format(context=build_context(chunk_ids), question=question)
    for attempt in range(retries):
        try:
            return llm.invoke(prompt).text.strip()
        except Exception as e:          # simple back-off for free-tier rate limits
            if attempt == retries - 1:
                return f"<LLM error: {e}>"
            time.sleep(10 * (attempt + 1))

In [8]:
def pipeline_1_baseline(query):
    # Pipeline 1: bi-encoder Top 10 -> take Top 3 directly -> LLM
    t0 = time.perf_counter()
    candidates = bi_encoder_retrieve(query)
    t1 = time.perf_counter()
    top3 = [cid for cid, _ in candidates[:TOP_K_CONTEXT]]
    answer = generate_answer(query, top3)
    t2 = time.perf_counter()
    return {"candidates": candidates, "top3": top3, "answer": answer,
            "t_retrieve": t1 - t0, "t_llm": t2 - t1}

def pipeline_2_rerank(query, candidates=None):
    # Pipeline 2: same Top 10 -> cross-encoder re-sort -> Top 3 -> LLM (same prompt)
    t0 = time.perf_counter()
    if candidates is None:
        candidates = bi_encoder_retrieve(query)
    t1 = time.perf_counter()
    reranked = cross_encoder_rerank(query, candidates)
    t2 = time.perf_counter()
    top3 = [r["chunk_id"] for r in reranked[:TOP_K_CONTEXT]]
    answer = generate_answer(query, top3)
    t3 = time.perf_counter()
    return {"reranked": reranked, "top3": top3, "answer": answer,
            "t_retrieve": t1 - t0, "t_rerank": t2 - t1, "t_llm": t3 - t2}

## 7. Failure-mode stress tests

Each test query has a **gold pattern**: a regex for the sentence that actually answers the question (found by reading the rulebook).
Any chunk that matches is marked ★ as a gold chunk, so the rank shift is measured, not eyeballed.

| # | Failure type | Query idea | Gold rule (page) |
|---|---|---|---|
| 1 | Exception / corner case | Does heavy armor slow a **dwarf**? | Dwarf trait: "speed is not reduced by wearing heavy armor" (p.15) |
| 2 | Negation & constraints | Does an **unconscious** creature *automatically fail* STR saves? | Unconscious condition (p.171) |
| 3 | Keyword trap / distractor | Healing magic having **no effect** | "no effect on undead or constructs" (Cure Wounds p.90, Heal p.97/100, Prayer of Healing p.102) |
| 4 | Buried at rank 4–10 + negation | Which creatures does Cure Wounds **not** work on? | same restriction inside the Cure Wounds spell text |
| 5 | Buried at rank 4–10 (partial) | Using a **bow** next to an enemy | "Ranged Attacks in Close Combat": disadvantage (p.76) |
| 6 | Limitation (re-ranker can't fix) | Knock an enemy out instead of killing it | "Knocking a Creature Out" (p.79) |

In [9]:
FAILURE_QUERIES = [
    {"type": "Exception / corner case",
     "query": "Can a dwarf's speed be reduced by heavy armor?",
     "gold": r"not reduced by wearing heavy armor"},
    {"type": "Negation & constraints",
     "query": "Is a creature that is unconscious automatically failing Strength saving throws?",
     "gold": r"automatically fails Strength and Dexterity"},
    {"type": "Keyword trap / distractor",
     "query": "What gives a spellcaster's healing magic no effect?",
     "gold": r"no effect on (undead or constructs|constructs or undead)"},
    {"type": "Buried at rank 4-10 (negation)",
     "query": "Which creatures does cure wounds not work on?",
     "gold": r"no effect on (undead or constructs|constructs or undead)"},
    {"type": "Buried at rank 4-10 (partial improvement)",
     "query": "Do I suffer any drawback for using a bow against an enemy standing right beside me?",
     "gold": r"within 5 feet of a hostile creature"},
    {"type": "Limitation: gold not in Top 10",
     "query": "How do I knock an enemy out instead of killing them?",
     "gold": r"Knocking a Creature Out"},
]

def is_gold(chunk_id, pattern):
    return re.search(pattern, chunks[chunk_id]["text"], re.I) is not None

def rank_table(reranked, gold):
    # one row per candidate: baseline rank vs cross-encoder rank (★ = contains the gold rule)
    df = pd.DataFrame(reranked).sort_values("baseline_rank")
    df["shift"] = df["baseline_rank"] - df["rerank_rank"]
    df["gold"] = df["chunk_id"].apply(lambda c: "★" if is_gold(c, gold) else "")
    df["page"] = df["chunk_id"].apply(lambda c: chunks[c]["page"])
    df["preview"] = df["chunk_id"].apply(lambda c: chunks[c]["text"][:90] + "…")
    return df[["baseline_rank", "rerank_rank", "shift", "gold", "chunk_id", "page",
               "bi_score", "ce_score", "preview"]].round(3)

In [10]:
from IPython.display import display, Markdown

results = []
for i, fq in enumerate(FAILURE_QUERIES, start=1):
    q, gold = fq["query"], fq["gold"]
    p1 = pipeline_1_baseline(q)
    p2 = pipeline_2_rerank(q, candidates=p1["candidates"])   # exactly the same Top 10

    table = rank_table(p2["reranked"], gold)
    gold_rows = table[table["gold"] == "★"]
    shifts = [f"chunk {r.chunk_id} (p.{r.page}): #{r.baseline_rank} → #{r.rerank_rank}"
              for r in gold_rows.itertuples()] or ["gold chunk NOT in baseline Top 10"]
    gold_in_p1 = any(is_gold(c, gold) for c in p1["top3"])
    gold_in_p2 = any(is_gold(c, gold) for c in p2["top3"])
    results.append({"#": i, "type": fq["type"], "query": q,
                    "gold rank shift": "; ".join(shifts),
                    "gold in P1 top3": gold_in_p1, "gold in P2 top3": gold_in_p2,
                    "answer_1": p1["answer"], "answer_2": p2["answer"],
                    "llm_ms_1": p1["t_llm"] * 1000, "llm_ms_2": p2["t_llm"] * 1000})

    display(Markdown(f"### Test {i}: {fq['type']}\n**Query:** {q}  \n**Gold rank shift:** {'; '.join(shifts)}  \n"
                     f"**Gold in Top 3?** baseline = `{gold_in_p1}`, re-ranked = `{gold_in_p2}`"))
    display(table.style.apply(lambda r: ["background-color: #fff3b0; color: #000000; font-weight: bold" if r["gold"] == "★" else "" for _ in r], axis=1)
                 .hide(axis="index"))
    display(Markdown(f"**Answer 1 (baseline, chunks {p1['top3']}):**\n> {p1['answer']}\n\n"
                     f"**Answer 2 (re-ranked, chunks {p2['top3']}):**\n> {p2['answer']}"))
    time.sleep(4)   # stay under the free-tier requests/minute limit

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


### Test 1: Exception / corner case
**Query:** Can a dwarf's speed be reduced by heavy armor?  
**Gold rank shift:** chunk 92 (p.15): #4 → #1  
**Gold in Top 3?** baseline = `False`, re-ranked = `True`

baseline_rank,rerank_rank,shift,gold,chunk_id,page,bi_score,ce_score,preview
1,2,-1,,316,46,0.443000,0.502000,Your class gives you proficiency with certain types of armor. If you wear armor that you l…
2,4,-2,,426,63,0.433000,-4.593000,"While pushing or dragging weight in excess of your carrying capacity, your speed drops to …"
3,5,-2,,326,47,0.431000,-4.611000,Only proficient warriors can manage their weight and bulk. Heavy armor doesn't let you add…
4,1,3,★,92,15,0.428000,3.974000,Ability Score Increase. Your Constitution score increases by 2. Age. Dwarves mature at the…
5,3,2,,88,15,0.415000,-3.846000,"Your hit point maximum increases by 1, and it increases by 1 every time you gain a level. …"
6,7,-1,,323,47,0.406000,-10.103000,"At the end of the battle, you can recover half your expended ammunition by taking a minute…"
7,10,-3,,318,46,0.386000,-10.906000,"Plate 1,500 gp 18 Str 15 Disadvantage 65 lb. Shield Shield 10 gp +2 — — 6 lb. Variant: Equ…"
8,6,2,,327,47,0.384000,-8.398000,"Flexible chain mail protects the joints. Plate. Plate consists of shaped, interlocking met…"
9,9,0,,324,47,0.380000,-10.715000,Hide. This crude armor consists of thick furs and pelts. It is commonly worn by barbarian …
10,8,2,,90,15,0.373000,-10.682000,"Dwarf Names A dwarf's name is granted by a clan elder, in accordance with tradition. Every…"


**Answer 1 (baseline, chunks [316, 426, 326]):**
> Based on the provided text, heavy armor reduces a wearer's speed by 10 feet if the Armor table shows a Strength requirement that the wearer does not meet. However, there is an exception: if you use the Variant: Encumbrance rules, you ignore the Strength column of the Armor table entirely. The context does not specifically mention dwarves.

**Answer 2 (re-ranked, chunks [92, 316, 88]):**
> According to the provided text, a dwarf's base walking speed of 25 feet is explicitly not reduced by wearing heavy armor. However, the text also notes the general rule that heavy armor reduces a wearer's speed by 10 feet unless the wearer meets the required Strength score listed on the Armor table.

### Test 2: Negation & constraints
**Query:** Is a creature that is unconscious automatically failing Strength saving throws?  
**Gold rank shift:** chunk 1089 (p.172): #5 → #2; chunk 1082 (p.171): #10 → #1  
**Gold in Top 3?** baseline = `False`, re-ranked = `True`

...

baseline_rank,rerank_rank,shift,gold,chunk_id,page,bi_score,ce_score,preview
1,6,-5,,184,27,0.449000,-9.480000,The weapon must have the two-handed or versatile property for you to gain this benefit. Pr…
2,3,-1,,533,76,0.398000,-6.108000,"When you attack a target that you can't see, you have disadvantage on the attack roll. Thi…"
3,7,-4,,564,80,0.390000,-9.792000,A ranged weapon attack automatically misses a target beyond the weapon's normal range. Eve…
4,8,-4,,183,27,0.370000,-9.995000,Choose one of the following options. You can't take a Fighting Style option more than once…
5,1,4,,1055,166,0.368000,-0.627000,Reduce the difficulty by one step if the characters have a benefit that their enemies don'…
6,9,-3,,563,80,0.364000,-10.406000,"In either case, if the mount provokes an opportunity attack while you're on it, the attack…"
7,5,2,,536,76,0.329000,-7.634000,Certain creatures (typically those larger than Medium) have melee attacks with a greater r…
8,4,4,★,535,76,0.325000,-7.478000,Ranged Attacks in Close Combat Aiming a ranged attack is more difficult when a foe is next…
9,10,-1,,176,26,0.313000,-11.041000,"Trained for Danger Not every member of the city watch, the village militia, or the queen's…"
10,2,8,,1056,166,0.306000,-4.330000,Situational benefits are similar to drawbacks except that they benefit the characters inst…


**Answer 1 (baseline, chunks [184, 533, 564]):**
> According to the provided text, when you make a ranged attack, the attack roll has disadvantage unless the weapon is a crossbow, a net, or a weapon that is thrown like a javelin (including a spear, trident, or dart). Since a bow is not listed among these exceptions, you suffer disadvantage on the attack roll. The context does not contain any other specific drawbacks for using a bow against an enemy standing beside you.

**Answer 2 (re-ranked, chunks [1055, 1056, 533]):**
> Based on the provided context, there is no mention of a drawback for using a bow against an enemy standing right beside you. The text only lists specific situational drawbacks such as the party being surprised, the enemy having cover, the characters being unable to see the enemy, environmental damage, or hindered mobility. Therefore, the context does not contain the answer.

### Test 6: Limitation: gold not in Top 10
**Query:** How do I knock an enemy out instead of killing them?  
**Gold rank shift:** gold chunk NOT in baseline Top 10  
**Gold in Top 3?** baseline = `False`, re-ranked = `False`

baseline_rank,rerank_rank,shift,gold,chunk_id,page,bi_score,ce_score,preview
1,1,0,,536,76,0.420000,1.758000,Certain creatures (typically those larger than Medium) have melee attacks with a greater r…
2,4,-2,,901,136,0.392000,-6.840000,"Actions Bite. Melee Weapon Attack: +5 to hit, reach 5 ft., one creature. Hit: 7 (1d8 + 3) …"
3,3,0,,842,123,0.381000,-6.587000,"Melee Weapon Attack: +5 to hit, reach 5 ft., one target. Hit: 10 (2d6 + 3) piercing damage…"
4,8,-4,,636,92,0.378000,-8.402000,You can use this telepathic link to issue commands to the creature while you are conscious…
5,7,-2,,879,131,0.375000,-8.126000,"Melee Weapon Attack: +5 to hit, reach 5 ft., one target. Hit: 10 (2d6 + 3) slashing damage…"
6,9,-3,,923,140,0.373000,-8.931000,"Claws. Melee Weapon Attack: +5 to hit, reach 5 ft., one target. Hit: 10 (2d6 + 3) slashing…"
7,2,5,,1055,166,0.373000,-1.760000,Reduce the difficulty by one step if the characters have a benefit that their enemies don'…
8,6,2,,840,123,0.370000,-7.812000,"Bite. Melee Weapon Attack: +4 to hit, reach 5 ft., one target. Hit: 5 (1d6 + 2) piercing d…"
9,5,4,,895,134,0.361000,-7.605000,"Melee Weapon Attack: +6 to hit, reach 10 ft., one target. Hit: 6 (1d4 + 4) piercing damage…"
10,10,0,,498,72,0.356000,-9.470000,The clatter of a sword striking against a shield. The terrible rending sound as monstrous …


**Answer 1 (baseline, chunks [536, 901, 842]):**
> Based on the provided context, there is no mention of how to knock an enemy out instead of killing them. The provided text only contains rules for unarmed strikes, opportunity attacks, and specific creature actions such as bites and webs.

**Answer 2 (re-ranked, chunks [536, 1055, 842]):**
> Based on the provided context, there is no mention of how to knock an enemy out instead of killing them.

### Summary of rank shifts

In [11]:
summary = pd.DataFrame(results)[["#", "type", "query", "gold rank shift", "gold in P1 top3", "gold in P2 top3"]]
summary

,#,type,query,gold rank shift,gold in P1 top3,gold in P2 top3
0,1,Exception / corner case,Can a dwarf's speed be reduced by heavy armor?,chunk 92 (p.15): #4 → #1,False,True
1,2,Negation & constraints,Is a creature that is unconscious automatically failing Strength saving throws?,chunk 1089 (p.172): #5 → #2; chunk 1082 (p.171): #10 → #1,False,True
2,3,Keyword trap / distractor,What gives a spellcaster's healing magic no effect?,chunk 676 (p.97): #3 → #3; chunk 695 (p.100): #6 → #2; chunk 709 (p.102): #8 → #1,True,True
3,4,Buried at rank 4-10 (negation),Which creatures does cure wounds not work on?,chunk 622 (p.90): #4 → #2,False,True
4,5,Buried at rank 4-10 (partial improvement),Do I suffer any drawback for using a bow against an enemy standing right beside me?,chunk 535 (p.76): #8 → #4,False,False
5,6,Limitation: gold not in Top 10,How do I knock an enemy out instead of killing them?,gold chunk NOT in baseline Top 10,False,False


In [12]:
# Side-by-side answers for the report
for r in results:
    print(f"=== Test {r['#']} [{r['type']}] {r['query']}")
    print("Answer 1 (baseline):", r["answer_1"])
    print("Answer 2 (rerank)  :", r["answer_2"], "\n")

=== Test 1 [Exception / corner case] Can a dwarf's speed be reduced by heavy armor?
Answer 1 (baseline): Based on the provided text, heavy armor reduces a wearer's speed by 10 feet if the Armor table shows a Strength requirement that the wearer does not meet. However, there is an exception: if you use the Variant: Encumbrance rules, you ignore the Strength column of the Armor table entirely. The context does not specifically mention dwarves.
Answer 2 (rerank)  : According to the provided text, a dwarf's base walking speed of 25 feet is explicitly not reduced by wearing heavy armor. However, the text also notes the general rule that heavy armor reduces a wearer's speed by 10 feet unless the wearer meets the required Strength score listed on the Armor table. 

=== Test 2 [Negation & constraints] Is a creature that is unconscious automatically failing Strength saving throws?
Answer 1 (baseline): Based on the provided context, there is no mention of whether an unconscious creature automati

## 8. Small retrieval evaluation (Hit@3 and MRR@10)

A wider set of rule questions (all with a hand-checked gold pattern) shows how often the right chunk ends up in the 3 passages the LLM actually sees.
No LLM calls here, only retrieval. Cases where re-ranking *hurts* are kept on purpose.

In [13]:
EVAL_SET = [
    ("Can a dwarf's speed be reduced by heavy armor?", r"not reduced by wearing heavy armor"),
    ("Is a creature that is unconscious automatically failing Strength saving throws?", r"automatically fails Strength and Dexterity"),
    ("What gives a spellcaster's healing magic no effect?", r"no effect on (undead or constructs|constructs or undead)"),
    ("Which creatures does cure wounds not work on?", r"no effect on (undead or constructs|constructs or undead)"),
    ("Do I suffer any drawback for using a bow against an enemy standing right beside me?", r"within 5 feet of a hostile creature"),
    ("How do I knock an enemy out instead of killing them?", r"Knocking a Creature Out"),
    ("Can a dwarf wear plate armor without the strength requirement slowing them?", r"not reduced by wearing heavy armor"),
    ("Can a paralyzed creature move or speak?", r"Paralyzed"),
    ("Can I attack twice with two weapons? What weapons are not allowed?", r"light melee weapon"),
    ("Can rangers or paladins use the Disengage action as a bonus action?", r"Cunning Action"),
    ("How far can my character leap across a gap?", r"Long Jump"),
    ("My character is drowning, how many rounds can they survive?", r"runs out of breath or is choking"),
    ("What happens to a hero who keeps marching for more than 8 hours a day?", r"Forced March"),
    ("Is there a penalty for attacking with a ranged weapon at long range?", r"beyond normal range"),
    ("Can a character wearing heavy armor they are not proficient with cast spells?", r"can't cast spells"),
    ("Can I take a short rest more than once? How many long rests can I take per day?", r"more than one long rest"),
    ("How long does it take to fully recover from exhaustion?", r"reduces a creature's exhaustion level by 1"),
    ("Can a creature hide while it is being watched?", r"can't hide from a creature that can see"),
    ("Can my character go without sleep, and what happens?", r"exhaustion"),
    ("How much damage do you take from falling?", r"1d6 bludgeoning damage for every 10 feet"),
]

def first_gold_rank(ids, pattern):
    return next((r for r, c in enumerate(ids, start=1) if is_gold(c, pattern)), None)

rows = []
for q, gold in EVAL_SET:
    cands = bi_encoder_retrieve(q)
    base_ids = [c for c, _ in cands]
    rr_ids = [r["chunk_id"] for r in cross_encoder_rerank(q, cands)]
    rows.append({"query": q,
                 "baseline_rank": first_gold_rank(base_ids, gold),
                 "rerank_rank": first_gold_rank(rr_ids, gold)})
eval_df = pd.DataFrame(rows)

def hit3(s): return (s.fillna(99) <= 3).mean()
def mrr(s):  return s.apply(lambda r: 0 if pd.isna(r) else 1 / r).mean()

eval_df["effect"] = eval_df.apply(
    lambda r: "not in top10" if pd.isna(r.baseline_rank) else
              ("improved" if r.rerank_rank < r.baseline_rank else
               "worse" if r.rerank_rank > r.baseline_rank else "same"), axis=1)
display(eval_df)
display(pd.DataFrame({
    "metric": ["Hit@3", "MRR@10", "Recall@10 (upper bound for both)"],
    "Pipeline 1 (baseline)": [hit3(eval_df.baseline_rank), mrr(eval_df.baseline_rank), eval_df.baseline_rank.notna().mean()],
    "Pipeline 2 (re-ranked)": [hit3(eval_df.rerank_rank), mrr(eval_df.rerank_rank), eval_df.rerank_rank.notna().mean()],
}).round(3))

,query,baseline_rank,rerank_rank,effect
0,Can a dwarf's speed be reduced by heavy armor?,4.0,1.0,improved
1,Is a creature that is unconscious automatically failing Strength saving throws?,5.0,1.0,improved
2,What gives a spellcaster's healing magic no effect?,3.0,1.0,improved
3,Which creatures does cure wounds not work on?,4.0,2.0,improved
4,Do I suffer any drawback for using a bow against an enemy standing right beside me?,8.0,4.0,improved
5,How do I knock an enemy out instead of killing them?,NaN,NaN,not in top10
6,Can a dwarf wear plate armor without the strength requirement slowing them?,9.0,4.0,improved
7,Can a paralyzed creature move or speak?,2.0,1.0,improved
8,Can I attack twice with two weapons? What weapons are not allowed?,7.0,4.0,improved
9,Can rangers or paladins use the Disengage action as a bonus action?,2.0,1.0,improved


,metric,Pipeline 1 (baseline),Pipeline 2 (re-ranked)
0,Hit@3,0.550,0.650
1,MRR@10,0.439,0.622
2,Recall@10 (upper bound for both),0.950,0.950


## 9. Latency profiling

Each stage is timed separately over several repeats, after a warm-up (the first GPU call includes CUDA initialisation and would distort the numbers).
LLM time is taken from the Gemini calls in section 7 (same model and prompt for both pipelines; it depends mostly on the network/API, not on the pipeline).

In [14]:
LATENCY_QUERIES = [fq["query"] for fq in FAILURE_QUERIES] + [q for q, _ in EVAL_SET[6:12]]
REPEATS = 5

# warm-up
for q in LATENCY_QUERIES[:2]:
    cross_encoder_rerank(q, bi_encoder_retrieve(q))
if DEVICE == "cuda":
    torch.cuda.synchronize()

retrieval_ms, rerank_ms = [], []
for _ in range(REPEATS):
    for q in LATENCY_QUERIES:
        t0 = time.perf_counter(); cands = bi_encoder_retrieve(q); t1 = time.perf_counter()
        cross_encoder_rerank(q, cands)
        if DEVICE == "cuda": torch.cuda.synchronize()
        t2 = time.perf_counter()
        retrieval_ms.append((t1 - t0) * 1000); rerank_ms.append((t2 - t1) * 1000)

# LLM timings: re-use the Gemini calls made during the stress tests (section 7)
llm_ms_p1 = [r["llm_ms_1"] for r in results]
llm_ms_p2 = [r["llm_ms_2"] for r in results]

def stats(x): return f"{statistics.mean(x):.1f} (p50 {statistics.median(x):.1f}, p95 {np.percentile(x, 95):.1f})"
total_p1 = [r for r in retrieval_ms]
total_p2 = [a + b for a, b in zip(retrieval_ms, rerank_ms)]

latency = pd.DataFrame([
    ["Bi-encoder query embedding + FAISS Top-10", stats(retrieval_ms), stats(retrieval_ms)],
    ["Cross-encoder re-rank (10 pairs)", "—", stats(rerank_ms)],
    ["Retrieval total (before LLM)", stats(total_p1), stats(total_p2)],
    ["LLM generation (Gemini API, network)", stats(llm_ms_p1), stats(llm_ms_p2)],
    ["End-to-end mean (retrieval + LLM)", f"{statistics.mean(total_p1) + statistics.mean(llm_ms_p1):.0f}",
     f"{statistics.mean(total_p2) + statistics.mean(llm_ms_p2):.0f}"],
], columns=["Stage (ms)", "Pipeline 1: Baseline", "Pipeline 2: Re-rank"])
print(f"Device: {DEVICE} | {len(retrieval_ms)} timed runs per stage")
print(f"Re-ranking overhead: +{statistics.mean(rerank_ms):.1f} ms per query "
      f"(~{statistics.mean(total_p2) / statistics.mean(total_p1):.1f}x the retrieval time)")
latency

Device: cuda | 60 timed runs per stage
Re-ranking overhead: +26.0 ms per query (~2.8x the retrieval time)


,Stage (ms),Pipeline 1: Baseline,Pipeline 2: Re-rank
0,Bi-encoder query embedding + FAISS Top-10,"14.7 (p50 14.0, p95 23.1)","14.7 (p50 14.0, p95 23.1)"
1,Cross-encoder re-rank (10 pairs),—,"26.0 (p50 26.0, p95 30.2)"
2,Retrieval total (before LLM),"14.7 (p50 14.0, p95 23.1)","40.7 (p50 40.6, p95 50.0)"
3,"LLM generation (Gemini API, network)","1270.0 (p50 1130.9, p95 1846.8)","1090.1 (p50 1022.3, p95 1394.3)"
4,End-to-end mean (retrieval + LLM),1285,1131


## 10. Scaling experiment: cross-encoding the whole corpus

To support the scaling question, time the cross-encoder over **all** chunks for a single query (what would happen without a bi-encoder first stage),
and compare it to the bi-encoder search over the same corpus. Then extrapolate linearly to bigger collections.

In [15]:
q = FAILURE_QUERIES[0]["query"]
all_pairs = [(q, c["text"]) for c in chunks]

t0 = time.perf_counter()
full_scores = cross_encoder.predict(all_pairs, batch_size=64)
if DEVICE == "cuda": torch.cuda.synchronize()
ce_full_s = time.perf_counter() - t0

t0 = time.perf_counter(); bi_encoder_retrieve(q, k=10); bi_s = time.perf_counter() - t0
per_pair_ms = ce_full_s / len(chunks) * 1000

print(f"Cross-encoder over all {len(chunks)} chunks: {ce_full_s*1000:.0f} ms  ({per_pair_ms:.2f} ms/pair)")
print(f"Bi-encoder + FAISS over all {len(chunks)} chunks: {bi_s*1000:.1f} ms")

# Did the full-corpus cross-encoder find the gold chunk too? (quality is good, cost is the problem)
best = np.argsort(-full_scores)[:3]
print("Full-corpus CE top-3 chunks:", list(best), "| contains gold:",
      any(is_gold(int(c), FAILURE_QUERIES[0]["gold"]) for c in best))

scale = pd.DataFrame({"corpus size (chunks)": [len(chunks), 10_000, 100_000, 1_000_000, 10_000_000]})
scale["cross-encoder on every chunk (s / query)"] = (scale.iloc[:, 0] * per_pair_ms / 1000).round(1)
scale["cross-encoder on Top-10 only (ms / query, measured in section 9)"] = round(statistics.mean(rerank_ms), 1)
scale

Cross-encoder over all 1113 chunks: 2318 ms  (2.08 ms/pair)
Bi-encoder + FAISS over all 1113 chunks: 11.8 ms
Full-corpus CE top-3 chunks: [np.int64(92), np.int64(1069), np.int64(316)] | contains gold: True


,corpus size (chunks),cross-encoder on every chunk (s / query),"cross-encoder on Top-10 only (ms / query, measured in section 9)"
0,1113,2.3,26.0
1,10000,20.8,26.0
2,100000,208.2,26.0
3,1000000,2082.3,26.0
4,10000000,20822.9,26.0


## 11. Notes for the report

All numbers needed for the four report sections are produced above:

- **Architecture comparison:** sections 4 & 5 (how each model is called).
- **Rank shifts & Answer 1 vs Answer 2:** section 7 tables + summary + printed answers.
- **Latency table:** section 9.
- **Scaling question:** section 10 (full-corpus cross-encoder cost vs. Top-10 only).

